#Spotify Hit Predictor 

##Data Cleaning 

**Objetive:** prepare a clean dataset of Spotify tracks to test wheter audio features predict hit potential.

**Source:** [Spotify Tracks Dataset](https://www.kaggle.com/datasets/yashdev01/spotify-tracks-dataset) — Kaggle.


In [1]:
import pandas as pd 

In [2]:
# Load the data 
df = pd.read_csv('../data/spotify-tracks-dataset.csv')                                          

In [3]:
#First look
print("Number of songs:", df.shape[0])
print("Number of columms:", df.shape[1])
print()
df.head()

Number of songs: 114000
Number of columms: 22



,Unnamed: 0.1,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,...,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,...,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,...,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,3,3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,...,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,4,4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,...,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


In [4]:
# Deleting columns 'Unndamed'
df = df.drop(columns=['Unnamed: 0.1', 'Unnamed: 0'])

#Now 20 columns
print('Number of columns now:', df.shape[1])
df.columns.tolist()
                                                          

Number of columns now: 20


['track_id',
 'artists',
 'album_name',
 'track_name',
 'popularity',
 'duration_ms',
 'explicit',
 'danceability',
 'energy',
 'key',
 'loudness',
 'mode',
 'speechiness',
 'acousticness',
 'instrumentalness',
 'liveness',
 'valence',
 'tempo',
 'time_signature',
 'track_genre']

# Checking data 


In [5]:
#Looking for NULL values 
print('NULL VALUES')
print(df.isnull().sum())

#Looking for duplicate values 
print('DUPLICATE VALUES')
print(df.duplicated().sum())

NULL VALUES
track_id            0
artists             1
album_name          1
track_name          1
popularity          0
duration_ms         0
explicit            0
danceability        0
energy              0
key                 0
loudness            0
mode                0
speechiness         0
acousticness        0
instrumentalness    0
liveness            0
valence             0
tempo               0
time_signature      0
track_genre         0
dtype: int64
DUPLICATE VALUES
450


**Findings:**

 The dataser has only 3 null values and 450 duplicates out of 114,000 songs. Only the 0.4%.

 The 3 nulls are concentrated in a single row missing `artists`, `album_name`
 and `track_name` — an incomplete record.

 The duplicates are likely the same `track_id` appearing across multiple genres
 rather than dirty data. Verified with `df['track_id'].duplicated().sum()`.


In [6]:
# Looking for any null values in all rows 
df[df.isnull().any(axis=1)]

#Deleting a row with null value
df = df.dropna()
print('Songs after cleaning null values:', df.shape[0])


Songs after cleaning null values: 113999


In [7]:
# Lets delete duplicate 
df = df.drop_duplicates()
print('Songs after delete duplicates:', df.shape[0])

Songs after delete duplicates: 113549


In [8]:
df[['popularity', 'duration_ms', 'tempo', 'energy', 'danceability', 'valence', 'loudness']].describe()

,popularity,duration_ms,tempo,energy,danceability,valence,loudness
count,113549.000000,1.135490e+05,113549.000000,113549.000000,113549.000000,113549.000000,113549.000000
mean,33.324433,2.280814e+05,122.175745,0.642091,0.567031,0.474205,-8.243408
std,22.283855,1.064131e+05,29.972954,0.251053,0.173409,0.259204,5.011422
min,0.000000,8.586000e+03,0.000000,0.000000,0.000000,0.000000,-49.531000
25%,17.000000,1.741840e+05,99.296000,0.473000,0.456000,0.260000,-9.998000
50%,35.000000,2.130000e+05,122.020000,0.685000,0.580000,0.464000,-6.997000
75%,50.000000,2.615880e+05,140.074000,0.854000,0.695000,0.683000,-5.001000
max,100.000000,5.237295e+06,243.372000,1.000000,0.985000,0.995000,4.532000


In [9]:
print('Total of songs with Tempo = 0:', (df['tempo'] == 0).sum())
print('Songs with less than 30 seconds:', (df['duration_ms'] < 30000).sum())
print('Songs with more than 10 minutes:', (df['duration_ms'] > 600000).sum())

Total of songs with Tempo = 0: 157
Songs with less than 30 seconds: 16
Songs with more than 10 minutes: 598


**Purpose:** confirm values make sense, not just that the data
types are correct. Using `describe()` is the fastest filter for semantically invalid
values that pass structural checks.

**Findings:**

- **`tempo` min = 0**: 157 rows. Zero BPM is physically impossible for music.
  This is not a song without rhythm. it is Spotify's tempo detection failing
  and encoding the failure as zero.

- **`duration_ms` range: 8.6 sec to 87 min**  16 rows under 30 seconds,
  598 rows over 10 minutes. Inspection of `track_genre` confirms most are
  DJ mixes and functional audio (white noise, sleep sounds) not songs.


### Inspecting long-duration tracks


In [10]:
# Finding tracks over 10 minutes 
long_songs = df[df['duration_ms'] > 600000]

# Count -> Genres that have tracks over 10 minutes 
print(long_songs['track_genre'].value_counts().head(15))

#Inspecting records 
songs_detailed = long_songs[['track_name', 'artists', 'track_genre', 'duration_ms', 'popularity']].copy()
songs_detailed['duration_min'] = (songs_detailed['duration_ms'] / 60000).round(1)

songs_detailed.sort_values('duration_ms', ascending=False).head(15)

track_genre
detroit-techno    58
iranian           51
new-age           41
classical         37
black-metal       35
chicago-house     29
folk              26
comedy            24
gospel            23
psych-rock        22
breakbeat         18
brazil            17
idm               15
minimal-techno    11
ambient           10
Name: count, dtype: int64


,track_name,artists,track_genre,duration_ms,popularity,duration_min
73617,Unity (Voyage Mix) Pt. 1,Tale Of Us,minimal-techno,5237295,35,87.3
10935,Crossing Wires 002 - Continuous DJ Mix,Timo Maas,breakbeat,4789026,11,79.8
24348,The Lab 03 - Continuous DJ Mix Part 1,Seth Troxler,detroit-techno,4730302,8,78.8
73840,Amnesia Ibiza Underground 10 DJ Mix,Loco Dice,minimal-techno,4563897,17,76.1
13344,House of Om - Mark Farina - Continuous Mix,Mark Farina,chicago-house,4447520,11,74.1
13245,Live In Tokyo - Continuous Mix,Mark Farina,chicago-house,4339826,11,72.3
13195,Greenhouse Construction,Mark Farina,chicago-house,4334721,12,72.2
27926,"NQ State of Mind, Vol. 1 - Continuous DJ Mix",Lenzman;Dan Stezo,drum-and-bass,4246206,15,70.8
101390,Ocean Waves Sounds,Ocean Sounds,sleep,4120258,39,68.7
45900,Internal Flight,Estas Tonne,guitar,3876276,34,64.6




- **Genre clustering:** the 598 tracks over 10 minutes concentrate in
  electronic genres (`detroit-techno` 58, `chicago-house` 29, `minimal-techno`
  11, `breakbeat` 18) and functional audio (`new-age` 41, `ambient` 10).

- **Track names are the decisive evidence.** The records self-identify:
  "Continuous DJ Mix", "Ocean Waves Sounds", "Vacuum Cleaner White Noise",
  "Electric Fan (Sound Masking Fan)", "Ruido Rosa Puro - Una Hora Versión".
  The longest record is 87 minutes — a DJ set, not a track.


### Removing invalid records 

 The `before` / `after` counters make the operation auditable. The notebook
 records how much data was removed, not just that something was removed.

In [11]:
before = df.shape[0]

#Keeping only songs with tempo 
df = df[df['tempo'] > 0]

#Kkeeping songs with a duration of 30 seconds to 10 minutes
df = df[(df['duration_ms'] >= 30000) & (df['duration_ms'] <= 600000)]

after = df.shape[0]
print(f'Numbers of songs before: {before:,}')
print(f'Number of songs after: {after:,}')
print(f'Deleted: {before - after:,}')


Numbers of songs before: 113,549
Number of songs after: 112,782
Deleted: 767


### Checking null values and genres 

In [12]:
#Looking for NULL values 
print('NULL VALUES')
print(df.isnull().sum())

#Looking for duplicate values 
print('DUPLICATE VALUES')
print(df.duplicated().sum())

NULL VALUES
track_id            0
artists             0
album_name          0
track_name          0
popularity          0
duration_ms         0
explicit            0
danceability        0
energy              0
key                 0
loudness            0
mode                0
speechiness         0
acousticness        0
instrumentalness    0
liveness            0
valence             0
tempo               0
time_signature      0
track_genre         0
dtype: int64
DUPLICATE VALUES
0


In [13]:
print("Genres:", df['track_genre'].nunique())

genres = sorted(df['track_genre'].unique())
print("Total:", len(genres), "genres\n")
for g in genres:
    print(g)

Genres: 114
Total: 114 genres

acoustic
afrobeat
alt-rock
alternative
ambient
anime
black-metal
bluegrass
blues
brazil
breakbeat
british
cantopop
chicago-house
children
chill
classical
club
comedy
country
dance
dancehall
death-metal
deep-house
detroit-techno
disco
disney
drum-and-bass
dub
dubstep
edm
electro
electronic
emo
folk
forro
french
funk
garage
german
gospel
goth
grindcore
groove
grunge
guitar
happy
hard-rock
hardcore
hardstyle
heavy-metal
hip-hop
honky-tonk
house
idm
indian
indie
indie-pop
industrial
iranian
j-dance
j-idol
j-pop
j-rock
jazz
k-pop
kids
latin
latino
malay
mandopop
metal
metalcore
minimal-techno
mpb
new-age
opera
pagode
party
piano
pop
pop-film
power-pop
progressive-house
psych-rock
punk
punk-rock
r-n-b
reggae
reggaeton
rock
rock-n-roll
rockabilly
romance
sad
salsa
samba
sertanejo
show-tunes
singer-songwriter
ska
sleep
songwriter
soul
spanish
study
swedish
synth-pop
tango
techno
trance
trip-hop
turkish
world-music


### Saving clean data 


In [14]:
df.to_csv('tracks_clean.csv', index=False)
print("Saved: tracks_clean.csv  with", df.shape[0], "songs")

Saved: tracks_clean.csv  with 112782 songs


In [15]:
print(df.dtypes)

track_id             object
artists              object
album_name           object
track_name           object
popularity            int64
duration_ms           int64
explicit               bool
danceability        float64
energy              float64
key                   int64
loudness            float64
mode                  int64
speechiness         float64
acousticness        float64
instrumentalness    float64
liveness            float64
valence             float64
tempo               float64
time_signature        int64
track_genre          object
dtype: object
